# Final Evaluation

The detailed comments explaining the pipeline **line by line** are provided in `notebooks/hparam-search.ipynb`.

This notebook serves a different purpose: to **fully exploit and stress-test the parameters of the best trial** obtained from the hyperparameter search.

Therefore, we do not repeat exhaustive inline documentation here. We only comment on points that are important to highlight for final evaluation (key decisions, relevant adjustments, and result interpretation).

# Libraries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
import pandas as pd
import random

from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, ProductTokenBuilder
from src.utils import TemporalSplitter, BlockBootstrapSampler

# Seeds

In [3]:
BASE_SEED = 42

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

# Seetings

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CKPT_DIR = RESULTS_DIR / "checkpoints" / "final_eval"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BEST_TRIAL_PATH = RESULTS_DIR / "best_trial_params.json"

# ── Data / Evaluation ─────────────────────────────────────────────
N_UPCS = 5
SMOOTH_WINDOW = 8
BETA_EDA = -2
K_NEIGHBORS = 4

# In this notebook, we incoporate the boostraping evaluation to 
# evaluate the performance/stability of the model. We go further in 
# this point above.

TRAIN_FRAC = 0.8          # final split for bootstrap
N_FOLDS = 5               # final temporal folds
EVAL_SEEDS = [11, 29, 42, 77, 123]  # Number of seeds to try
# Number of bootstraps to run. We get 20 version of the model 
# with different (bootstrap) samples of the training data.
N_BOOTSTRAP = 20
# Block size for the bootstrap. Instead of using a bootstrap
# sample (individual weeks) of the training data, we use a block bootstrap,
# to keep the local temporal structure of the data.
BLOCK_SIZE = 4      
# Minimum training fraction. We use at least 50% of the original data
# for each boostrap sample. We try to avoid a complete resampling
# of the data.
MIN_TRAIN_FRAC = 0.5

# ── Training ───────────────────────────────────────────────────────
N_EPOCHS_P0 = 350
N_EPOCHS_P1 = 350
N_EPOCHS_P2 = 400
PATIENCE    = 30
ES_PATIENCE = 70

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Load best trial ───────────────────────────────────────────────
with open(BEST_TRIAL_PATH, "r", encoding="utf-8") as f:
    best_trial = json.load(f)

HIDDEN_OPTIONS = {
    "64_32":        (64, 32),
    "128_64":       (128, 64),
    "192_96":       (192, 96),
    "256_128":      (256, 128),
    "256_128_64":   (256, 128, 64),
}

# We load the best trial parameters.
params = best_trial["params"]

# We extract the parameters.
N_KNOTS = int(params["N_KNOTS"])
HIDDEN = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
ACT = "gelu"
DROPOUT = float(params["DROPOUT"])
D_STORE = 16
D_BRAND = 8
D_STYLE = 8

BATCH_SIZE = int(params["BATCH_SIZE"])
LR_P0 = float(params["LR_P0"])
LR_P1 = float(params["LR_P1"])
LAMBDA_SMOOTH = float(params["LAMBDA_SMOOTH"])
LAMBDA_POS = float(params["LAMBDA_POS"])
LAMBDA_CROSS_ALPHA = float(params["LAMBDA_CROSS_ALPHA"])
LAMBDA_CROSS_U = float(params["LAMBDA_CROSS_U"])
print("Best trial loaded:")
print(json.dumps(best_trial, indent=2, ensure_ascii=False))

Device: cuda
Best trial loaded:
{
  "trial": 0,
  "robust_score": 0.5534630604560247,
  "mean_r2": 0.5048785691243625,
  "std_r2": 0.0,
  "mean_elast_score": 0.4858449133166223,
  "std_elast_score": 0.0,
  "params": {
    "N_KNOTS": 7,
    "HIDDEN_KEY": "128_64",
    "DROPOUT": 0.08783947930743292,
    "LR_P0": 0.0002794098312439086,
    "LR_P1": 6.23523937203338e-05,
    "LAMBDA_SMOOTH": 5.2396889573852424e-05,
    "LAMBDA_POS": 0.0013227702079602346,
    "LAMBDA_CROSS_ALPHA": 5.635029491198898e-05,
    "LAMBDA_CROSS_U": 0.10880753631969278,
    "BATCH_SIZE": 256
  }
}


# Loader

In [5]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")
print(f"Dataset shape: {df.shape}")

encoder = ColumnEncoder()

_, store_cats = encoder.factorize(df, "store_code",        sort=True)
_, week_cats  = encoder.factorize(df, "week_id",           sort=True)
_, brand_cats = encoder.factorize(df, "brand_family_norm", sort=True)
_, style_cats = encoder.factorize(df, "style_segment_norm",sort=True)

n_stores = len(store_cats)
n_weeks  = len(week_cats)
n_brands = len(brand_cats)
n_styles = len(style_cats)

brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"]  = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

print(f"Stores: {n_stores}  |  Weeks: {n_weeks}")

mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS)

full_wide_raw = mp_builder.transform().copy()
n_upcs = mp_builder.n

print(f"Full wide shape: {full_wide_raw.shape}")
print(f"UPCs selected: {n_upcs}")
print(f"Top {N_UPCS}: {mp_builder.selected_upcs[:N_UPCS]}")

Dataset shape: (463722, 44)
Stores: 70  |  Weeks: 302
Full wide shape: (19808, 171)
UPCs selected: 5
Top 5: [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]


# Neighbors

In [6]:
# Static Metadata per UPC position (same for all batches)
upc_meta = (
    df.groupby("upc_code")[["category_code", "brand_family_norm", "style_segment_norm", "liters_per_upc"]]
    .first()
    .loc[mp_builder.selected_upcs]
)

# category_code can be string → factorize for comparability
cat_codes, _ = pd.factorize(upc_meta["category_code"], sort=True)

neighbor_meta = {
    "category": torch.tensor(cat_codes, dtype=torch.long, device=device),
    "brand": torch.tensor(upc_meta["brand_family_norm"].values, dtype=torch.long, device=device),
    "style": torch.tensor(upc_meta["style_segment_norm"].values, dtype=torch.long, device=device),
    "liters": torch.tensor(upc_meta["liters_per_upc"].values, dtype=torch.float32, device=device),
}

# Temporal Folds

In [7]:
splitter = TemporalSplitter(week_col="week_id")
fold_splits = splitter.expanding_splits(
    df=full_wide_raw,
    n_folds=N_FOLDS,
    min_train_frac=MIN_TRAIN_FRAC,
)

print(f"N folds available: {len(fold_splits)}")
for i, (train_fold, val_fold) in enumerate(fold_splits):
    print(
        f"Fold {i}: train={len(train_fold):,} "
        f"val={len(val_fold):,} "
        f"train_weeks={train_fold['week_id'].nunique()} "
        f"val_weeks={val_fold['week_id'].nunique()}"
    )

N folds available: 1
Fold 0: train=9,756 val=10,052 train_weeks=151 val_weeks=151


# Functions

In [8]:
store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}

# In the nootebook hparam-search.ipynb is called build_fold_frames, but
# this one is not introduced the smooth_window parameter. We take it directly
# from the global variables.
def prepare_fold_frames(train_wide: pd.DataFrame, val_wide: pd.DataFrame):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
    val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()

    for i in range(n_upcs):
        col = f"log_liters_{i}"
        for df_w in [train_wide_s, val_wide_s]:
            df_w[col] = (
                df_w.groupby("store_code")[col]
                .transform(lambda s: s.rolling(window=SMOOTH_WINDOW, min_periods=1).mean())
            )

    return train_wide, val_wide, train_wide_s, val_wide_s

# This function is called build_fold_datasets in the notebook hparam-search.ipynb
# but here we call it build_loaders because it's included the DataLoaderFactory
# object. Therefore, we build the train and validation dataset and the loaders in
# one function.
def build_loaders(train_wide, val_wide, train_wide_s, val_wide_s):
    loader_factory = DataLoaderFactory(num_workers=4, pin_memory=True, persistent_workers=True)

    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs)
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs)
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs)
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs)

    train_loader_p0 = loader_factory.create_train_loader(
        train_ds_p0, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    val_loader_p0 = loader_factory.create_eval_loader(
        val_ds_p0, batch_size=BATCH_SIZE, shuffle=False
    )

    train_loader = loader_factory.create_train_loader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    val_loader = loader_factory.create_eval_loader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False
    )

    return train_loader_p0, val_loader_p0, train_loader, val_loader

In [9]:
# This function is new compared to the notebook hparam-search.ipynb
# because we need to build the model components in a function.
# We encapsulate the model components in an unique function.
def build_model_components(train_wide: pd.DataFrame):
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        x_i = train_wide[f"log_price_{i}"].values
        config = builder.build_from_data(x_i, n_knots=N_KNOTS, q_min=0.05, q_max=0.95)
        spline_configs.append(config)

    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"]  for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]   for cfg in spline_configs])
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    token_builder = ProductTokenBuilder(
        n=n_upcs,
        n_stores=n_stores,
        d_store=D_STORE,
        n_brands=n_brands,
        d_brand=D_BRAND,
        n_styles=n_styles,
        d_style=D_STYLE,
    )

    def make_model(enforce_negative_beta, use_cross, device):
        head = IntegrableDemandHead(
            context_dim=token_builder.d_token,
            K_splines=N_KNOTS,
            n=n_upcs,
            k_neighbors=K_NEIGHBORS,
            hidden=HIDDEN,
            act=ACT,
            dropout=DROPOUT,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        model = ICDN(
            context_builder=token_builder,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)
        return model

    return make_model

In [10]:
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, neighbor_meta, phase_name=""):

    best_val_loss = float("inf")
    no_improve    = 0
    # We use a dictionary to store the history of the training.
    history       = {"train_loss": [], "val_loss": [], "val_mae": [], "lr": []}
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    print(f"\n{'='*70}")
    print(f"  {phase_name}  |  {n_epochs} epochs  |  ckpt: {ckpt_path.name}")
    print(f"{'='*70}")

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        total_loss, total_denom = 0.0, 0.0

        for batch in train_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true   = batch["demands"]  
            obs_mask = batch["obs_mask"] 

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, eps_hat, aux = model(batch, return_parts=True, neighbor_meta=neighbor_meta)
                    loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                        aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                        aux["pairs"], aux["alpha"])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, eps_hat, aux = model(batch, return_parts=True, neighbor_meta=neighbor_meta)
                loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                    aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                    aux["pairs"], aux["alpha"])
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            denom = obs_mask.sum().item()
            total_loss  += logs["loss"].item() * denom
            total_denom += denom

        train_loss = total_loss / max(total_denom, 1.0)

        # ── Val ────────────────────────────────────────────────────
        model.eval()
        total_loss, total_denom = 0.0, 0.0
        total_abs,  total_mask  = 0.0, 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                y_true   = batch["demands"]  
                obs_mask = batch["obs_mask"] 

                y_hat, eps_hat, aux = model(batch, return_parts=True, neighbor_meta=neighbor_meta)
                _, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                aux["pairs"], aux["alpha"])

                denom = obs_mask.sum().item()
                total_loss  += logs["loss"].item() * denom
                total_denom += denom

                total_abs  += ((y_hat - y_true).abs() * obs_mask).sum().item()
                total_mask += obs_mask.sum().item()

        val_loss = total_loss / max(total_denom, 1.0)
        val_mae  = total_abs  / max(total_mask, 1.0)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(val_mae)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        scheduler.step(val_loss)
        
        # In this point, there's an important difference between this notebook
        # and the hparam-search.ipynb. In this case, we don't reset the patience
        # counter.
        # if new_lr < prev_lr:     (X)   
        #     no_improve = 0       (X)
        # The reason is that this parameters are already optimized therefore
        # we don't need to be so strict with the patience.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
            saved = "x"
        else:
            no_improve += 1
            saved = ""

        if (epoch + 1) % 10 == 0 or no_improve == 0:
            print(f"Epoch {epoch+1:4d}/{n_epochs}"
                  f"  train={train_loss:.4f}"
                  f"  val={val_loss:.4f}"
                  f"  mae={val_mae:.4f}"
                  f"  {saved}")

        if no_improve >= es_patience:
            print(f"  Early stopping in epoch {epoch+1}")
            break

    print(f"\nBest val_loss: {best_val_loss:.4f} - {ckpt_path.name}")
    return history

In [11]:
# We add these helpers because they are often used in the
# execution of the notebook. In fact, we could have added them
# to the hparam-search notebook too.

def freeze_nonlinear(model):
    # Freeze the own splines (head_w) and the cross splines (head_cross)
    # but leave the linear cross term (head_alpha) free
    for attr in ["head_w", "head_cross"]:
        head = getattr(model.head.param_head, attr)
        head.weight.requires_grad_(False)
        head.bias.requires_grad_(False)
        
def unfreeze_nonlinear(model):
    for attr in ["head_w", "head_cross"]:
        head = getattr(model.head.param_head, attr)
        head.weight.requires_grad_(True)
        head.bias.requires_grad_(True)

def init_beta_prior(model, beta_target):
    beta_raw_init = torch.log(
        torch.exp(torch.tensor(-beta_target, dtype=torch.float32)) - 1.0
    )
    with torch.no_grad():
        model.head.param_head.head_beta.weight.zero_()
        model.head.param_head.head_beta.bias.fill_(beta_raw_init)
    print(f"beta initialized: target={beta_target:.3f}  beta_raw_init={beta_raw_init:.4f}")

print("Helpers defined")

Helpers defined


In [12]:
def compute_global_metrics(model, val_loader):
    model.eval()
    all_y_hat, all_y_true, all_mask = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_hat, _, _ = model(batch, return_parts=True, neighbor_meta=neighbor_meta)
            y_true   = batch["demands"]
            obs_mask = batch["obs_mask"]  

            all_y_hat.append(y_hat.cpu())
            all_y_true.append(y_true.cpu())
            all_mask.append(obs_mask.cpu())

    y_hat = torch.cat(all_y_hat)
    y_true = torch.cat(all_y_true)
    mask = torch.cat(all_mask)

    mae = float(((y_hat - y_true).abs() * mask).sum() / mask.sum())
    rmse = float(torch.sqrt((((y_hat - y_true) ** 2) * mask).sum() / mask.sum()))

    preds_np = y_hat.numpy()
    targets_np = y_true.numpy()
    mask_np = mask.bool().numpy()

    ss_res = ((targets_np[mask_np] - preds_np[mask_np]) ** 2).sum()
    ss_tot = ((targets_np[mask_np] - targets_np[mask_np].mean()) ** 2).sum()
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2,
    }

# This function extract the elasticity rows from the model.
def extract_elasticity_rows(model, val_loader, run_type, run_id, fold=None, seed=None, bootstrap_run=None):
    model.eval() # We set the model to evaluation mode.
    rows = []

    with torch.no_grad(): # We don't need to compute the gradients.
        for batch in val_loader:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            # ids[:, 0] holds the store_code (label-encoded contiguous index).
            store_idx  = batch["ids"][:, 0].cpu().numpy()
            # We recover the store codes from the store indices.
            # Recall that they were encoded as integers.
            # Namely, [101, 123, 103] <-> [0, 1, 2]
            store_code = np.array(store_cats)[store_idx]
            # obs_mask is already (B, n) — pre-stacked in MultiProductDataset.__init__
            obs_mask = batch["obs_mask"]
            y_hat, _, aux = model.run(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
            E = aux["E"].cpu().numpy()
            mask_np = obs_mask.cpu().numpy().astype(bool)
            # demands is already (B, n) — pre-stacked in MultiProductDataset.__init__
            y_true_np = batch["demands"].cpu().numpy()
            y_hat_np = y_hat.cpu().numpy()

            # Elasticity tensor shape: (B, n_upcs, n_upcs)
            B = E.shape[0]
            # We recover the UPC names. We extract them from the 
            # MultiProductBuilder object.
            upc_names = np.array(mp_builder.selected_upcs)

            for b in range(B): # We iterate over the batch.
                sc = store_code[b] # We get the store code.
                for i in range(n_upcs): # We iterate over the products.
                    if not mask_np[b, i]: # We skip the products that are not observed.
                        continue
                    for j in range(n_upcs): # We iterate over the products again.
                        if not mask_np[b, j]: # We skip the products that are not observed.
                            continue
                        # We append the elasticity row to the list.
                        rows.append({
                            "run_type": run_type,
                            "run_id": run_id,
                            "fold": fold,
                            "seed": seed,
                            "bootstrap_run": bootstrap_run,
                            "store_code": sc,
                            "upc_i": upc_names[i],
                            "upc_j": upc_names[j],
                            "type": "own" if i == j else "cross",
                            "E": E[b, i, j],
                            "y_true_i": y_true_np[b, i],
                            "y_hat_i": y_hat_np[b, i],
                        })

    return pd.DataFrame(rows)

In [13]:
def train_one_run(train_fold, val_fold, seed, run_type, run_id, fold=None, bootstrap_run=None):
    # We set the seeds for reproducibility.
    set_all_seeds(seed)

    # We prepare the frames for the train and validation folds.
    train_wide, val_wide, train_wide_s, val_wide_s = prepare_fold_frames(train_fold, val_fold)

    # We build the loaders for the train and validation folds.
    train_loader_p0, val_loader_p0, train_loader, val_loader = build_loaders(
        train_wide, val_wide, train_wide_s, val_wide_s
    )

    # We build the model components.
    make_model = build_model_components(train_wide)

    # We define the checkpoint paths.
    ckpt_p0 = CKPT_DIR / f"{run_type}_{run_id}_seed{seed}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"{run_type}_{run_id}_seed{seed}_phase1.pt"

    # ── PHASE 0 ─────────────────────────────────────────────────────
    # The goal of this phase is to obtain a robust initialization before
    # unlocking the model's full flexibility. To do so:
    #
    #   1. Cross-price effects are available (use_cross=True) and the spline
    #      weights are frozen (head_w → zeros, requires_grad=False). This reduces
    #      the model to a log-linear demand: 
    #      log(q_i) \approx b + beta·log(p_i) + sum_{j\neq i}·alpha_{ij}·log(p_j) log(p_i) .
    #
    #   2. The head_beta bias is initialized with the inverse softplus of
    #      BETA_EDA, so that the own-price elasticity at startup equals exactly
    #      -BETA_EDA. This gives the model an economically sensible starting
    #      point instead of a random one.
    #
    #   3. The loss applies no smoothness or positivity penalties (lambda_smooth=0,
    #      lambda_pos=0): only the demand prediction error is minimized.
    #
    # By the end of this phase, beta and b are well calibrated, which makes
    # convergence easier in later phases when spline weights and cross-price
    # effects are unfrozen.

    model_p0 = make_model(enforce_negative_beta=True, use_cross=True, device=device)

    # We freeze the spline weights.
    freeze_nonlinear(model_p0)

    # We initialize the beta bias.
    init_beta_prior(model_p0, BETA_EDA)
    with torch.no_grad():
        model_p0.head.param_head.head_alpha.weight.zero_()
        model_p0.head.param_head.head_alpha.bias.zero_()

    # We define the loss function.
    loss_p0 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=0,
        lambda_pos=0,
        lambda_cross_alpha=LAMBDA_CROSS_ALPHA, 
        lambda_cross_u=0, 
        reduction="mean"
    )
    decay, no_decay = [], []
    for name, p in model_p0.named_parameters():
        if not p.requires_grad: continue
        if ("head_w" in name) or ("head_cross" in name) or ("head_alpha" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=LR_P0,
    )
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    run_training(
        model=model_p0,
        train_loader=train_loader_p0,
        val_loader=val_loader_p0,
        loss_fn=loss_p0,
        optimizer=opt_p0,
        scheduler=sch_p0,
        n_epochs=N_EPOCHS_P0,
        es_patience=ES_PATIENCE,
        ckpt_path=ckpt_p0,
        device=device,
        neighbor_meta=neighbor_meta,
        phase_name=f"{run_type} | {run_id} | seed={seed} | P0",
    )

    # ── Phase 1: Unlock spline weights with smoothed targets ───────────────────
    # Building on the stable beta and b from Phase 0, this phase introduces the
    # spline flexibility that was previously frozen:
    #
    #   1. The model is initialized from the Phase 0 checkpoint. The spline
    #      weights (head_w) are unfrozen (requires_grad=True), allowing the
    #      model to learn non-linear price responses beyond the log-linear baseline.
    #
    #   2. The loss applies smoothness or positivity penalties
    #      (lambda_smooth\neq0, lambda_pos\neq0): pure fit to the training data.
    #
    #   3. Training uses the non-smoothed data (train_loader / val_loader),
    #      unlike Phase 0 which trained on rolling-average targets.
    #
    # By the end of this phase, the spline shapes are well fit to the raw demand
    # signal, providing a good initialization for the cross-price phase that follows.

    model_p1 = make_model(enforce_negative_beta=True, use_cross=True, device=device)
    model_p1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    unfreeze_nonlinear(model_p1)

    loss_p1 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=LAMBDA_SMOOTH,
        lambda_pos=LAMBDA_POS,
        lambda_cross_alpha=LAMBDA_CROSS_ALPHA,
        lambda_cross_u=LAMBDA_CROSS_U,
        reduction="mean"
    )
    decay, no_decay = [], []
    for name, p in model_p1.named_parameters():
        if not p.requires_grad: continue
        if ("head_w" in name) or ("head_cross" in name) or ("head_alpha" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=LR_P1,
    )
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    run_training(
        model=model_p1,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_p1,
        optimizer=opt_p1,
        scheduler=sch_p1,
        n_epochs=N_EPOCHS_P1,
        es_patience=ES_PATIENCE,
        ckpt_path=ckpt_p1,
        device=device,
        neighbor_meta=neighbor_meta,
        phase_name=f"{run_type} | {run_id} | seed={seed} | P1",
    )

    metrics = compute_global_metrics(model_p1, val_loader)
    df_e = extract_elasticity_rows(
        model=model_p1,
        val_loader=val_loader,
        run_type=run_type,
        run_id=run_id,
        fold=fold,
        seed=seed,
        bootstrap_run=bootstrap_run,
    )

    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)

    return metrics, df_e

# Training - Folds and Seeds

In [14]:
fold_metrics_rows = []
fold_elasticity_rows = []

for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
    for seed in EVAL_SEEDS:
        print(f"\n=== Fold {fold_id} | Seed {seed} ===")

        metrics, df_e = train_one_run(
            train_fold=train_fold,
            val_fold=val_fold,
            seed=seed,
            run_type="kfold",
            run_id=f"fold{fold_id}_seed{seed}",
            fold=fold_id,
            bootstrap_run=None,
        )

        fold_metrics_rows.append({
            "fold": fold_id,
            "seed": seed,
            "n_train": len(train_fold),
            "n_val": len(val_fold),
            **metrics,
        })

        fold_elasticity_rows.append(df_e)

nn_kfold_metrics_raw = pd.DataFrame(fold_metrics_rows)
nn_kfold_elasticities_raw = pd.concat(fold_elasticity_rows, ignore_index=True)

print("K-fold + seeds completed")


=== Fold 0 | Seed 11 ===
beta initialized: target=-1.000  beta_raw_init=0.5413

  kfold | fold0_seed11 | seed=11 | P0  |  350 epochs  |  ckpt: kfold_fold0_seed11_seed11_phase0.pt
Epoch    1/350  train=2.5964  val=3.0976  mae=3.5735  x
Epoch    2/350  train=0.8652  val=1.3655  mae=1.8096  x
Epoch    3/350  train=0.4604  val=0.7439  mae=1.1436  x
Epoch    4/350  train=0.3034  val=0.5098  mae=0.8751  x
Epoch    7/350  train=0.2012  val=0.3410  mae=0.6727  x
Epoch    9/350  train=0.1630  val=0.3303  mae=0.6569  x
Epoch   10/350  train=0.1497  val=0.3040  mae=0.6290  x
Epoch   12/350  train=0.1307  val=0.2478  mae=0.5479  x
Epoch   15/350  train=0.1113  val=0.2398  mae=0.5434  x
Epoch   16/350  train=0.1067  val=0.1491  mae=0.4089  x
Epoch   20/350  train=0.0922  val=0.1864  mae=0.4825  
Epoch   29/350  train=0.0727  val=0.1345  mae=0.3932  x
Epoch   30/350  train=0.0703  val=0.1251  mae=0.3743  x
Epoch   40/350  train=0.0592  val=0.1348  mae=0.4052  
Epoch   50/350  train=0.0513  val=0.10

# Fold - Summary

In [15]:
nn_kfold_metrics_summary = pd.DataFrame([{
    "mae_val_mean": nn_kfold_metrics_raw["mae_val"].mean(),
    "mae_val_std": nn_kfold_metrics_raw["mae_val"].std(ddof=1),
    "rmse_val_mean": nn_kfold_metrics_raw["rmse_val"].mean(),
    "rmse_val_std": nn_kfold_metrics_raw["rmse_val"].std(ddof=1),
    "r2_val_mean": nn_kfold_metrics_raw["r2_val"].mean(),
    "r2_val_std": nn_kfold_metrics_raw["r2_val"].std(ddof=1),
    "n_runs": len(nn_kfold_metrics_raw),
}])

display(nn_kfold_metrics_summary)

,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std,n_runs
0,0.72598,NaN,0.921718,NaN,0.362514,NaN,1


In [16]:
df_e = nn_kfold_elasticities_raw.copy()
# --- Own elasticities ---
own_elast_summary = (
    df_e[df_e["type"] == "own"]
    .groupby(["store_code", "upc_i"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", lambda s: s.quantile(0.025)),
        elasticity_ci_high=("E", lambda s: s.quantile(0.975)),
        n_obs=("E", "size"),
    )
    .rename(columns={"upc_i": "upc_code"})
    .sort_values(["store_code", "upc_code"])
    .reset_index(drop=True)
)
# --- Cross elasticities ---
cross_elast_summary = (
    df_e[df_e["type"] == "cross"]
    .groupby(["store_code", "upc_i", "upc_j"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", lambda s: s.quantile(0.025)),
        elasticity_ci_high=("E", lambda s: s.quantile(0.975)),
        n_obs=("E", "size"),
    )
    .sort_values(["store_code", "upc_i", "upc_j"])
    .reset_index(drop=True)
)
display(own_elast_summary.head())
display(cross_elast_summary.head())

,store_code,upc_code,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,n_obs
0,5,3410017306,-4.839558,2.182045,-9.598160,-2.562508,69
1,5,7289000011,0.286729,0.642599,-0.666863,1.104033,88
2,8,1820000784,-4.695794,2.210494,-7.917576,-0.803628,151
3,8,3410010505,-0.682598,1.217785,-5.830562,-0.193746,151
4,8,3410017306,-5.190039,2.329510,-9.965625,-1.768370,69


,store_code,upc_i,upc_j,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,n_obs
0,5,3410017306,7289000011,-0.246218,0.245800,-0.872481,-0.058386,56
1,5,7289000011,3410017306,0.079017,0.056522,-0.056413,0.147786,56
2,8,1820000784,3410010505,0.068040,0.094087,0.003113,0.185870,151
3,8,1820000784,3410017306,0.218576,0.154003,0.009280,0.507051,69
4,8,1820000784,7289000011,0.554992,0.591427,-0.012872,1.722316,140


# Training - Bootstrapping

In [17]:
# Now the final train and validation sets are created without folds,
# we split them into train and validation sets directly.
train_final, val_final = splitter.single_split(full_wide_raw, train_frac=TRAIN_FRAC)
train_weeks_final = sorted(train_final["week_id"].unique())

print(f"Train final: {len(train_final):,} rows | {len(train_weeks_final)} weeks")
print(f"Val final:   {len(val_final):,} rows | {val_final['week_id'].nunique()} weeks")

Train final: 15,822 rows | 241 weeks
Val final:   3,986 rows | 61 weeks


In [18]:
# We create the bootstrap sampler. 
# To visualize:
# ─────────────────────────────────────────────────────────────────────────────
# Suppose we have 10 training weeks and block_size=4.
#
#   train_weeks  = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
#   block_starts = [0, 4, 8]   (non-overlapping block start indices)
#   n_blocks     = 3
#
# We draw n_blocks indices WITH replacement:
#   sampled_indices = [2, 0, 2]    (block 2 drawn twice, block 1 never drawn)
#
#   idx=2  start=8: weeks [9, 10]       (short tail block, < block_size)
#   idx=0  start=0: weeks [1, 2, 3, 4]
#   idx=2  start=8: weeks [9, 10]       (repeated)
#
# Bootstrap sample contains weeks: [9,10,1,2,3,4, 9,10]
#   · Weeks 5-8 are absent. The model never sees them in this run.
#   · Weeks 9-10 appear twice. Their observations are counted double.
#
# Repeating this N_BOOTSTRAP times gives N_BOOTSTRAP slightly different
# training sets, producing a distribution of model outputs from which
# we can estimate uncertainty (confidence intervals, std of elasticities).
# ─────────────────────────────────────────────────────────────────────────────

bootstrap_sampler = BlockBootstrapSampler(
    week_col="week_id",
    block_size=BLOCK_SIZE,
    rng=np.random.default_rng(BASE_SEED),
)

In [19]:
BOOTSTRAP_TRAINING_SEED = EVAL_SEEDS[0]

bootstrap_metrics_rows = []
bootstrap_elasticity_rows = []

for b in range(N_BOOTSTRAP):
    print(f"\n=== Bootstrap {b+1}/{N_BOOTSTRAP} ===")

    train_bs = bootstrap_sampler.sample(train_final, train_weeks_final)

    metrics, df_e = train_one_run(
        train_fold=train_bs,
        val_fold=val_final,
        seed=BOOTSTRAP_TRAINING_SEED,
        run_type="bootstrap",
        run_id=f"bootstrap{b}",
        fold=None,
        bootstrap_run=b,
    )

    bootstrap_metrics_rows.append({
        "bootstrap_run": b,
        "seed": BOOTSTRAP_TRAINING_SEED,
        "n_train": len(train_bs),
        "n_val": len(val_final),
        **metrics,
    })

    bootstrap_elasticity_rows.append(df_e)

nn_bootstrap_metrics_raw = pd.DataFrame(bootstrap_metrics_rows)
nn_bootstrap_elasticities_raw = pd.concat(bootstrap_elasticity_rows, ignore_index=True)

print("Bootstrap completed")
display(nn_bootstrap_metrics_raw.head())


=== Bootstrap 1/20 ===
beta initialized: target=-1.000  beta_raw_init=0.5413

  bootstrap | bootstrap0 | seed=11 | P0  |  350 epochs  |  ckpt: bootstrap_bootstrap0_seed11_phase0.pt
Epoch    1/350  train=2.1477  val=0.8531  mae=1.2818  x
Epoch    2/350  train=0.5852  val=0.4265  mae=0.7771  x
Epoch    3/350  train=0.3528  val=0.2581  mae=0.5902  x
Epoch    6/350  train=0.1995  val=0.1907  mae=0.5062  x
Epoch    7/350  train=0.1786  val=0.1219  mae=0.3969  x
Epoch    9/350  train=0.1548  val=0.0796  mae=0.3141  x
Epoch   10/350  train=0.1451  val=0.2030  mae=0.5526  
Epoch   14/350  train=0.1181  val=0.0753  mae=0.3156  x


KeyboardInterrupt: 

# Bootstrapping - Summary

In [ ]:
# Defining the quantile functions.
def q025(x): return np.percentile(x, 2.5)
def q975(x): return np.percentile(x, 97.5)

# Own-elasticity summary.
nn_bootstrap_own_summary = (
    nn_bootstrap_elasticities_raw[nn_bootstrap_elasticities_raw["type"] == "own"]
    .groupby(["store_code", "upc_i"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", q025),
        elasticity_ci_high=("E", q975),
        n_obs=("E", "count"),
    )
    .rename(columns={"upc_i": "upc_code"})
)

# Cross-elasticity summary.
nn_bootstrap_cross_summary = (
    nn_bootstrap_elasticities_raw[nn_bootstrap_elasticities_raw["type"] == "cross"]
    .groupby(["store_code", "upc_i", "upc_j"], as_index=False)
    .agg(
        elasticity_mean=("E", "mean"),
        elasticity_std=("E", "std"),
        elasticity_ci_low=("E", q025),
        elasticity_ci_high=("E", q975),
        n_obs=("E", "count"),
    )
)

display(nn_bootstrap_own_summary.head())
display(nn_bootstrap_cross_summary.head())

,store_code,upc_code,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,n_obs
0,5,7289000011,-1.041464,0.265896,-1.519826,-0.542125,280
1,8,1820000784,-0.969109,0.282802,-1.465303,-0.417385,1220
2,8,3410010505,-0.766034,0.211984,-1.168639,-0.359830,1220
3,8,7289000011,-1.113261,0.304544,-1.648992,-0.527848,1220
4,9,1820000784,-0.707713,0.277156,-1.242676,-0.266385,1040


,store_code,upc_i,upc_j,elasticity_mean,elasticity_std,elasticity_ci_low,elasticity_ci_high,n_obs
0,8,1820000784,3410010505,0.040486,0.061077,-0.077498,0.157551,1220
1,8,1820000784,7289000011,0.035893,0.060362,-0.085325,0.152554,1220
2,8,3410010505,1820000784,0.053612,0.020458,0.012402,0.089636,1220
3,8,3410010505,7289000011,0.053836,0.018824,0.017036,0.089315,1220
4,8,7289000011,1820000784,-0.005338,0.135094,-0.241231,0.254183,1220


In [ ]:
# Bootstrapping - Metrics Summary
nn_bootstrap_metrics_summary = pd.DataFrame([{
    "mae_val_mean": nn_bootstrap_metrics_raw["mae_val"].mean(),
    "mae_val_std": nn_bootstrap_metrics_raw["mae_val"].std(ddof=1),
    "rmse_val_mean": nn_bootstrap_metrics_raw["rmse_val"].mean(),
    "rmse_val_std": nn_bootstrap_metrics_raw["rmse_val"].std(ddof=1),
    "r2_val_mean": nn_bootstrap_metrics_raw["r2_val"].mean(),
    "r2_val_std": nn_bootstrap_metrics_raw["r2_val"].std(ddof=1),
    "n_bootstrap_runs": len(nn_bootstrap_metrics_raw),
}])

display(nn_bootstrap_metrics_summary)

,mae_val_mean,mae_val_std,rmse_val_mean,rmse_val_std,r2_val_mean,r2_val_std,n_bootstrap_runs
0,0.503822,0.027694,0.640672,0.027714,0.449522,0.049217,20


# Save Results

In [ ]:
nn_kfold_elasticities_raw.to_csv(DATA_DIR / "nn_kfold_elasticities_raw.csv", index=False)
nn_kfold_metrics_raw.to_csv(DATA_DIR / "nn_kfold_metrics_raw.csv", index=False)
nn_bootstrap_elasticities_raw.to_csv(DATA_DIR / "nn_bootstrap_elasticities_raw.csv", index=False)

print("Saved:")
print("- nn_kfold_elasticities_raw.csv")
print("- nn_kfold_metrics_raw.csv")
print("- nn_bootstrap_elasticities_raw.csv")

Guardado:
- nn_kfold_metrics_raw.csv
- nn_kfold_metrics_summary.csv
- nn_kfold_elasticities_raw.csv
- nn_bootstrap_metrics_raw.csv
- nn_bootstrap_metrics_summary.csv
- nn_bootstrap_elasticities_raw.csv
- nn_bootstrap_own_summary.csv
- nn_bootstrap_cross_summary.csv
